# Capítulo 6 — SELECT avançado: Ver os seus dados com novos olhos

**Livro:** começa na página **223** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(6, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Combinando dados existentes · PDF p. 226

#### Exercício: Uma coluna de categoria

A tabela `filmes` tem uma coluna T/F para cada gênero. Acrescente `categoria VARCHAR(10)`.

In [ ]:
%%sql
ALTER TABLE filmes ADD COLUMN categoria VARCHAR(15);
SELECT titulo, drama, comedia, acao, terror, ficcao, infantil, desenho, categoria FROM filmes;

## Povoando sua nova coluna · PDF p. 226

#### Exercício: Vários UPDATEs

Preencha `categoria` com um `UPDATE` por gênero: `drama`, `comédia`, `ação`, `terror`, `ficção`, `família` (infantil) e `desenho`. Em que ordem você rodou? Um filme com dois gêneros fica com qual?

In [ ]:
%%sql
UPDATE filmes SET categoria = 'drama' WHERE drama = 'T';
UPDATE filmes SET categoria = 'comédia' WHERE comedia = 'T';
UPDATE filmes SET categoria = 'ação' WHERE acao = 'T';
UPDATE filmes SET categoria = 'terror' WHERE terror = 'T';
UPDATE filmes SET categoria = 'ficção' WHERE ficcao = 'T';
UPDATE filmes SET categoria = 'família' WHERE infantil = 'T';
UPDATE filmes SET categoria = 'desenho' WHERE desenho = 'T';
SELECT titulo, categoria FROM filmes;

## UPDATE com uma expressão CASE · PDF p. 229

#### Exercício: Um único UPDATE com CASE

Refaça com **um** `UPDATE` e `CASE`. A primeira condição verdadeira vence. Use `ELSE 'diversos'`. Mude a ordem: desenho primeiro, depois família.

In [ ]:
%%sql
UPDATE filmes SET categoria =
    CASE
        WHEN desenho = 'T' THEN 'desenho'
        WHEN infantil = 'T' THEN 'família'
        WHEN drama = 'T' THEN 'drama'
        WHEN comedia = 'T' THEN 'comédia'
        WHEN acao = 'T' THEN 'ação'
        WHEN terror = 'T' THEN 'terror'
        WHEN ficcao = 'T' THEN 'ficção'
        ELSE 'diversos'
    END;
SELECT titulo, categoria FROM filmes;

## Parece que temos um problema · PDF p. 231

#### Exercício: CASE com mais de uma condição

Filmes de terror com classificação `16` devem virar `terror leve` e os de `18`, `terror`. Faça com `CASE` dentro de `UPDATE ... WHERE terror = 'T'`.

In [ ]:
%%sql
UPDATE filmes SET categoria =
    CASE WHEN classificacao = '18' THEN 'terror' ELSE 'terror leve' END
WHERE terror = 'T';
SELECT titulo, classificacao, categoria FROM filmes WHERE terror = 'T';

## Tente um simples ORDER BY · PDF p. 239

#### Exercício: ORDER BY

Liste título e categoria dos filmes em ordem alfabética de título.

In [ ]:
%%sql
SELECT titulo, categoria FROM filmes ORDER BY titulo;

## ORDER com duas colunas · PDF p. 243

#### Exercício: Ordenando por duas colunas

Ordene por categoria e, dentro de cada categoria, por título.

In [ ]:
%%sql
SELECT categoria, titulo FROM filmes ORDER BY categoria, titulo;

## Inverta o comando ORDER com DESC · PDF p. 246

#### Exercício: DESC

Liste os filmes da classificação mais alta para a mais baixa e, empatados, em ordem de título.

In [ ]:
%%sql
SELECT classificacao, titulo FROM filmes ORDER BY classificacao DESC, titulo;

## SUM pode somá-las para nós · PDF p. 250

#### Exercício: SUM

Quanto a `Nina` vendeu de biscoitos no total?

In [ ]:
%%sql
SELECT SUM(vendas) AS total_nina FROM vendas_biscoitos WHERE primeiro_nome = 'Nina';

## SUM (some) todas de uma vez com GROUP BY · PDF p. 251

#### Exercício: GROUP BY

Mostre o total de cada vendedora, da maior para a menor. Quem é a líder?

In [ ]:
%%sql
SELECT primeiro_nome, SUM(vendas) AS total
FROM vendas_biscoitos
GROUP BY primeiro_nome
ORDER BY total DESC;

## AVG combinada com GROUP BY · PDF p. 252

#### Exercício: AVG

Mostre a média por vendedora, arredondada em 2 casas.

In [ ]:
%%sql
SELECT primeiro_nome, ROUND(AVG(vendas), 2) AS media
FROM vendas_biscoitos GROUP BY primeiro_nome ORDER BY media DESC;

## MIN e MAX · PDF p. 253

#### Exercício: MIN e MAX

Mostre a menor e a maior venda de cada uma.

In [ ]:
%%sql
SELECT primeiro_nome, MIN(vendas) AS menor, MAX(vendas) AS maior
FROM vendas_biscoitos GROUP BY primeiro_nome ORDER BY primeiro_nome;

## COUNT (conte) os dias · PDF p. 253

#### Exercício: COUNT

Em quantos dias cada uma vendeu **alguma coisa** (`vendas > 0`)? E em quantos dias diferentes houve venda no total (`COUNT(DISTINCT data)`)? Aqui, a primeira.

In [ ]:
%%sql
SELECT primeiro_nome, COUNT(data) AS dias_com_venda
FROM vendas_biscoitos WHERE vendas > 0
GROUP BY primeiro_nome ORDER BY dias_com_venda DESC;

## SELECT DISTINCT (selecione diferentes) valores · PDF p. 255

#### Exercício: DISTINCT

Liste as datas distintas em que houve vendas, em ordem.

In [ ]:
%%sql
SELECT DISTINCT data FROM vendas_biscoitos ORDER BY data;

## LIMIT (limite) o número de resultados · PDF p. 257

#### Exercício: LIMIT

Mostre só as 2 primeiras do ranking de totais.

In [ ]:
%%sql
SELECT primeiro_nome, SUM(vendas) AS total
FROM vendas_biscoitos GROUP BY primeiro_nome ORDER BY total DESC LIMIT 2;

## LIMIT (limite) apenas para a segunda colocada · PDF p. 258

#### Exercício: LIMIT com OFFSET

Mostre **só a segunda colocada**. No MySQL seria `LIMIT 1, 1`; aqui é `LIMIT 1 OFFSET 1`.

In [ ]:
%%sql
SELECT primeiro_nome, SUM(vendas) AS total
FROM vendas_biscoitos GROUP BY primeiro_nome ORDER BY total DESC LIMIT 1 OFFSET 1;